# Task 2.1 — Feature audit

This notebook audits the HDF5 data product produced by the final C++ extractor. It works on the 7k-row test file or the full ~1.4M-row file.

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/work/clas12b/users/skuditha/ALERT/alert_pid/python')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.feature_audit import *

In [8]:
# Set these paths before running.
H5_PATHS = [
    #'/work/clas12b/users/skuditha/ALERT/alert_pid/data/small.h5',
    '/work/clas12b/users/skuditha/ALERT/alert_pid/data/file1.h5',
]
LABEL_MAP_PATH = '/work/clas12b/users/skuditha/ALERT/alert_pid/config/label_map.json'
OUTDIR = Path('/work/clas12b/users/skuditha/ALERT/alert_pid/reports/feature_audit')

print('Edit H5_PATHS first.')

Edit H5_PATHS first.


In [9]:
# Load dataset
ds = load_audit_dataset(H5_PATHS, LABEL_MAP_PATH)
print({'n_rows': ds.n_rows, 'n_features': ds.n_features, 'files': [str(p) for p in ds.paths]})
print(ds.feature_names)

{'n_rows': 4879, 'n_features': 38, 'files': ['/work/clas12b/users/skuditha/ALERT/alert_pid/data/file1.h5']}
['px', 'py', 'pz', 'p', 'pt', 'theta', 'phi', 'vx', 'vy', 'vz', 'vr', 'v3', 'n_hits', 'sum_adc', 'path', 'dEdx', 'dedx_recomputed', 'p_drift', 'sum_residuals', 'residual_per_hit', 'adc_per_hit', 'tof_time', 'pathlength', 'cluster_x', 'cluster_y', 'cluster_z', 'cluster_energy', 'n_bar', 'n_wedge', 'beta', 'm2', 'log_p', 'log_pt', 'log_sum_adc', 'log_path', 'log_dEdx', 'log_dedx_recomputed', 'log_cluster_energy']


In [10]:
class_balance = compute_class_balance(ds)
feature_summary = compute_feature_summary(ds)
mask_summary = compute_mask_summary(ds)
unit_sanity = infer_unit_sanity(ds)
pathologies = detect_pathologies(ds)
separation = compute_separation_table(ds)
pair_focus = pair_focus_summary(ds)

class_balance

,class_index,class_name,count,fraction
0,0,proton,709,0.145317
1,1,deuteron,903,0.185079
2,2,triton,886,0.181595
3,3,helium3,1199,0.245747
4,4,helium4,1182,0.242263


In [11]:
feature_summary.head(20)

,feature,valid_count,invalid_count,valid_fraction,raw_min,raw_max,valid_min,valid_max,valid_mean,valid_std,zeros_in_stored_values
0,m2,4765,114,0.976635,0.000000,3.297920e+08,300.792419,3.297920e+08,9.538776e+06,1.904845e+07,114
1,adc_per_hit,4879,0,1.000000,45.125000,3.119750e+03,45.125000,3.119750e+03,8.507393e+02,6.328883e+02,0
2,beta,4879,0,1.000000,0.062512,1.195031e+00,0.062512,1.195031e+00,3.702197e-01,2.285781e-01,0
3,cluster_energy,4879,0,1.000000,0.393653,2.976904e+01,0.393653,2.976904e+01,8.134883e+00,6.269028e+00,0
4,cluster_x,4879,0,1.000000,-89.876656,8.987666e+01,-89.876656,8.987666e+01,-1.149991e+00,6.313926e+01,0
5,cluster_y,4879,0,1.000000,-89.876656,8.987666e+01,-89.876656,8.987666e+01,5.378138e-01,6.202351e+01,0
6,cluster_z,4879,0,1.000000,-284.496307,2.852483e+02,-284.496307,2.852483e+02,-2.927957e+00,8.300657e+01,0
7,dEdx,4879,0,1.000000,2.145970,3.606563e+02,2.145970,3.606563e+02,7.948277e+01,6.507429e+01,0
8,dedx_recomputed,4879,0,1.000000,2.145970,3.606563e+02,2.145970,3.606563e+02,7.948277e+01,6.507429e+01,0
9,log_cluster_energy,4879,0,1.000000,-0.932286,3.393469e+00,-0.932286,3.393469e+00,1.674120e+00,1.050157e+00,0


In [12]:
mask_summary

,metric,value
0,rows_with_any_masked_feature,114.000000
1,rows_with_no_masked_feature,4765.000000
2,mean_invalid_features_per_row,0.023365
3,max_invalid_features_in_row,1.000000


In [13]:
unit_sanity

,check,value,comment
0,p_median,593.811462,"Large O(1) suggests GeV/c, O(100-1000) suggest..."
1,tof_time_median_ns,1.297752,Expected ns-scale positive cluster timing.
2,pathlength_median,114.017540,Check whether pathlength looks mm-scale rather...
3,beta_median_stored,0.302645,Should be comfortably below 1 for most rows.
4,beta_median_recomputed_mm_ns,0.302645,Recomputed with c = 299.792458 mm/ns.
5,frac_beta_gt_1p0_stored,0.023365,Diagnostic only; no row cuts in audit.
6,frac_beta_gt_1p0_recomputed,0.023365,High value flags a unit mismatch or timing pat...
7,frac_m2_negative,0.000000,"Negative m2 can occur, but large fractions des..."


In [14]:
pathologies

,pathology,count
0,nonfinite_stored_values,0
1,rows_with_any_nonfinite_stored_value,0
2,valid_time_le_zero,0
3,valid_pathlength_le_zero,0
4,valid_p_le_zero,0
5,valid_dEdx_le_zero,0
6,valid_cluster_energy_le_zero,0
7,valid_beta_le_zero,0
8,valid_beta_gt_1p2,0


In [15]:
separation.head(15)

,feature,fisher_score
0,log_sum_adc,1.831717
1,log_dEdx,1.594467
2,log_dedx_recomputed,1.594467
3,adc_per_hit,1.510532
4,cluster_energy,1.492365
5,sum_adc,1.359148
6,log_cluster_energy,1.320308
7,dEdx,1.041789
8,dedx_recomputed,1.041789
9,residual_per_hit,0.121958


In [16]:
pair_focus

,feature,class_name,count,median,p16,p84
0,p,deuteron,903,6.600452e+02,395.679618,1.202609e+03
1,p,helium4,1182,5.615690e+02,385.106599,9.747220e+02
2,tof_time,deuteron,903,1.266937e+00,0.661264,2.199258e+00
3,tof_time,helium4,1182,1.386436e+00,0.792498,2.176845e+00
4,pathlength,deuteron,903,1.140175e+02,91.082382,1.548419e+02
5,pathlength,helium4,1182,1.140175e+02,91.082382,1.548419e+02
6,dEdx,deuteron,903,2.602068e+01,11.176964,6.060740e+01
7,dEdx,helium4,1182,1.240358e+02,79.453745,1.975429e+02
8,cluster_energy,deuteron,903,3.422640e+00,1.045350,5.884483e+00
9,cluster_energy,helium4,1182,1.423071e+01,8.266706,1.974312e+01


In [17]:
corr = compute_correlation_matrix(ds)
high_corr = high_correlation_pairs(corr, threshold=0.95)
high_corr.head(30)

,feature_a,feature_b,corr
0,log_dEdx,log_dedx_recomputed,1.000000
1,dEdx,dedx_recomputed,1.000000
2,p,p_drift,0.999999
3,sum_residuals,residual_per_hit,0.991776
4,path,log_path,0.984437
5,sum_adc,adc_per_hit,0.981926
6,log_sum_adc,log_dedx_recomputed,0.954874
7,log_sum_adc,log_dEdx,0.954874


In [18]:
OUTDIR.mkdir(parents=True, exist_ok=True)
plot_feature_histograms(ds, KEY_PHYSICS_FEATURES, OUTDIR / 'key_histograms')
plot_feature_histograms(ds, DEFAULT_FEATURE_NAMES, OUTDIR / 'histograms')
plot_scatter_by_class(ds, 'p', 'beta', OUTDIR / 'beta_vs_p.png')
plot_scatter_by_class(ds, 'p', 'm2', OUTDIR / 'm2_vs_p.png')
plot_scatter_by_class(ds, 'p', 'dEdx', OUTDIR / 'dEdx_vs_p.png')
plot_scatter_by_class(ds, 'pathlength', 'cluster_energy', OUTDIR / 'cluster_energy_vs_pathlength.png')
plot_correlation_heatmap(corr, OUTDIR / 'correlation_heatmap.png')
print(f'Plots written under {OUTDIR.resolve()}')

Plots written under /ceph24/hallb/clas12/users/skuditha/ALERT/alert_pid/reports/feature_audit


In [ ]:
# One-shot batch run
# tables = run_full_feature_audit(H5_PATHS, LABEL_MAP_PATH, OUTDIR)